In [368]:
import pandas as pd
from nb_utils import set_root
import numpy as np
import sys
import json
import os
from pathlib import Path
from typing import List, Union


PROJECT_DIR = set_root(2)

In [369]:
path_data = PROJECT_DIR / "data"
file_path_horm = path_data / "03_primary" / "teste.csv"
file_path_metrics = path_data / "03_primary" / "metrics.parquet"

In [370]:
# Carregar o arquivo Parquet
df_horm = pd.read_csv(file_path_horm)
df_metrics = pd.read_parquet(file_path_metrics)
df_metrics['ID'] = df_metrics['ID'].astype(int)
df_horm['ID'] = df_horm['ID'].astype(int)
df_merged = pd.merge(df_metrics, df_horm, on='ID', how='left')

In [475]:
# Classificando os tipos de espermatozoides com base em porcentagens
def classify_sperm(row):
    # Tipo A: Rápidos e progressivos
    if(row['VCL'] >= row['VAP'] and row['VAP'] >= row['VSL']):

        if (row['VCL'] == 0 or row['VAP'] == 0 or row['VSL'] == 0) and row['MAD'] == 0:  # Imóveis
            return "Tipo D"
        
        elif row['VCL'] >= row['VSL'] * 8:  # Hiperativos: VCL 40% maior que VAP
            return "Tipo Hiperativo"
        
        elif ((round(row['VCL'] - row['VAP'], 0)  <= row['VSL'] * 0.1) # Diferença menor que 10% de VAP
            and (round(row['VAP'] - row['VSL'], 0)  <= row['VSL'] * 0.1) and row['VSL'] >= 40):  
            return "Tipo A"
        
        # Tipo B: Rápidos mas menos progressivos (VCL > VAP > VSL, mas com mais variação entre eles)
        elif (( round(row['VCL'] - row['VAP'], 0)  <= row['VSL'] * 0.1 ) and row['VSL'] > 20):
            return "Tipo B"
        
        # Tipo C: Movimentos fracos (VSL e/ou VAP baixos)
        elif (((row['VAP'] - row['VSL']) <= row['VSL'] * 0.5) ):  
            return "Tipo C"
        
        # Se não se encaixar em nenhuma das classificações acima
        else:
            return "Não Classificado"

    else:
            return "Mal Identificado"

# Supondo que 'df_merged' seja o DataFrame que contém as colunas VCL, VAP, VSL
df = df_merged.copy()

# Aplicando a função para criar a nova coluna
df['tipo_espermatozoide'] = df.apply(classify_sperm, axis=1)

# Verificando os resultados
print(df.groupby(['tipo_espermatozoide']).size())

tipo_espermatozoide
Mal Identificado    17661
Não Classificado    23635
Tipo A                152
Tipo B              15292
Tipo C              14661
Tipo D              11193
Tipo Hiperativo      6706
dtype: int64


In [478]:
df = df[(df['tipo_espermatozoide'] != 'Não Classificado') & (df['tipo_espermatozoide'] != 'Mal Identificado')]

df.groupby(['tipo_espermatozoide']).size()
df.shape

(48004, 75)

In [479]:
# Agrupar os dados por ID e calcular as quantidades de cada tipo
df_grouped = df.groupby('ID').agg({
      'Age (years)': 'first',
      'DNA fragmentation index, DFI (%)': 'first',
      'Body mass index (kg/m²)': 'first',
      'High DNA stainability, HDS (%)': 'first',
      'Serum follicle-stimulating hormone, FSH (IU/L)':'first',
      'Serum anti-Müllerian hormone, AMH (pmol/L)': 'first',
      'Seminal plasma anti-Müllerian hormone (AMH) (pmol/L)': 'first'
      
          # Pega a primeira ocorrência da idade para cada ID
}).reset_index()

# Contar a quantidade de cada tipo separadamente e adicioná-las como novas colunas
df_grouped['Tipo_A'] = df['tipo_espermatozoide'].eq('Tipo A').groupby(df['ID']).sum().values
df_grouped['Tipo_B'] = df['tipo_espermatozoide'].eq('Tipo B').groupby(df['ID']).sum().values
df_grouped['Tipo_C'] = df['tipo_espermatozoide'].eq('Tipo C').groupby(df['ID']).sum().values
df_grouped['Tipo_D'] = df['tipo_espermatozoide'].eq('Tipo D').groupby(df['ID']).sum().values
df_grouped['Hiperativo'] = df['tipo_espermatozoide'].eq('Tipo Hiperativo').groupby(df['ID']).sum().values

# Visualizar o resultado
df_grouped.to_csv("df_grouped.csv")


In [454]:
import pandas as pd
import plotly.express as px

# Agrupar por faixas etárias
bins = [0, 20, 30, 40, 50, 60, 100]  # Ajuste as faixas etárias conforme necessário
labels = ['0-20', '21-30', '31-40', '41-50', '51-60', '60+']
df_grouped['Age_Group'] = pd.cut(df_grouped['Age (years)'], bins=bins, labels=labels, right=False)

# Somar os tipos de espermatozoides por faixa etária
df_age_group = df_grouped.groupby('Age_Group')[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum().reset_index()

# Calcular o total de espermatozoides por faixa etária
df_age_group['Total'] = df_age_group[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum(axis=1)

# Calcular a proporção de cada tipo
for col in ['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']:
    df_age_group[f'{col}_Proporcao'] = df_age_group[col] / df_age_group['Total']

# Selecionar apenas as colunas de proporção para o gráfico
df_proporcao = df_age_group[['Age_Group', 'Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao']]

# Criar o gráfico de barras empilhadas com proporções
fig = px.bar(df_proporcao, 
             x='Age_Group', 
             y=['Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao'], 
             title="Proporção de tipos de espermatozoides por faixa etária",
             labels={"Age_Group": "Faixa Etária", "value": "Proporção", "variable": "Tipo"},
             barmode='stack')

# Mostrar o gráfico
fig.show()


C:\Users\ccana\AppData\Local\Temp\ipykernel_23064\3312771824.py:10: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [457]:
import pandas as pd
import plotly.express as px

# Carregar os dados
df = pd.read_csv('df_grouped.csv')

# Definir as faixas de FSH
bins = [0, 5, 10, 15, 20, float('inf')]
labels = ['0-5', '5-10', '10-15', '15-20', '20+']
df['FSH_Group'] = pd.cut(df['Serum follicle-stimulating hormone, FSH (IU/L)'], bins=bins, labels=labels, right=False)

# Somar os tipos de espermatozoides por faixa de FSH
df_fsh_group = df.groupby('FSH_Group')[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum().reset_index()

# Calcular o total de espermatozoides por faixa de FSH
df_fsh_group['Total'] = df_fsh_group[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum(axis=1)

# Calcular a proporção de cada tipo
for col in ['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']:
    df_fsh_group[f'{col}_Proporcao'] = df_fsh_group[col] / df_fsh_group['Total']

# Selecionar apenas as colunas de proporção para o gráfico
df_proporcao = df_fsh_group[['FSH_Group', 'Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao']]

# Criar o gráfico de barras empilhadas com proporções
fig = px.bar(df_proporcao, 
             x='FSH_Group', 
             y=['Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao'], 
             title="Proporção de tipos de espermatozoides por faixa de Serum follicle-stimulating hormone (FSH)",
             labels={"FSH_Group": "Faixa de FSH (IU/L)", "value": "Proporção", "variable": "Tipo"},
             barmode='stack')

# Mostrar o gráfico
fig.show()


C:\Users\ccana\AppData\Local\Temp\ipykernel_23064\2509220353.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
import pandas as pd
import plotly.express as px

# Carregar os dados
df = pd.read_csv('df_grouped.csv')

# Verificar se o nome da coluna está correto (se necessário, ajuste o nome da coluna)
df.columns  # Verifique os nomes das colunas para garantir que 'DNA fragmentation index, DFI (%)' está correto

# Definir as faixas de FSH
bins = [0, 11, 15, 23, 80, float('inf')]
labels = ['0-11', '11-15', '15-23', '23-30', '30+']
df['FSH_Group'] = pd.cut(df['DNA fragmentation index, DFI (%)'], bins=bins, labels=labels, right=False)

# Somar os tipos de espermatozoides por faixa de FSH
df_fsh_group = df.groupby('FSH_Group')[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum().reset_index()

# Calcular o total de espermatozoides por faixa de FSH
df_fsh_group['Total'] = df_fsh_group[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum(axis=1)

# Calcular a proporção de cada tipo
for col in ['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']:
    df_fsh_group[f'{col}_Proporcao'] = df_fsh_group[col] / df_fsh_group['Total']

# Selecionar apenas as colunas de proporção para o gráfico
df_proporcao = df_fsh_group[['FSH_Group', 'Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao']]

# Criar o gráfico de barras empilhadas com proporções
fig = px.bar(df_proporcao, 
             x='FSH_Group', 
             y=['Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao'], 
             title="Proporção de tipos de espermatozoides por DNA fragmentation index, DFI (%)",
             labels={"FSH_Group": "Faixa de DFI (%)", "value": "Proporção", "variable": "Tipo"},
             barmode='stack')

# Mostrar o gráfico
fig.show()


C:\Users\ccana\AppData\Local\Temp\ipykernel_23064\1164697345.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [486]:
print(df_grouped['DNA fragmentation index, DFI (%)'].mean())
df_grouped.describe()


18.811764705882354


,ID,Age (years),"DNA fragmentation index, DFI (%)",Body mass index (kg/m²),"High DNA stainability, HDS (%)","Serum follicle-stimulating hormone, FSH (IU/L)","Serum anti-Müllerian hormone, AMH (pmol/L)",Seminal plasma anti-Müllerian hormone (AMH) (pmol/L),Tipo_A,Tipo_B,Tipo_C,Tipo_D,Hiperativo
count,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000,85.000000
mean,43.000000,36.447059,18.811765,30.230588,10.752941,4.349306,49.411765,312.329412,1.788235,179.905882,172.482353,131.682353,78.894118
std,24.681302,10.006552,13.005946,7.801633,8.179396,2.372823,28.147272,730.856826,2.144173,183.912804,212.788830,150.703368,59.879248
min,1.000000,22.000000,3.000000,18.800000,2.000000,1.000000,10.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.000000
25%,22.000000,29.000000,11.000000,24.900000,6.000000,2.600000,31.000000,11.000000,0.000000,30.000000,20.000000,25.000000,39.000000
50%,43.000000,34.000000,15.000000,29.200000,8.000000,4.100000,44.000000,43.000000,1.000000,102.000000,93.000000,81.000000,63.000000
75%,64.000000,43.000000,23.000000,33.700000,13.000000,5.001000,61.000000,248.000000,3.000000,298.000000,245.000000,191.000000,102.000000
max,85.000000,61.000000,80.000000,62.700000,60.000000,14.200000,176.000000,4105.000000,8.000000,701.000000,1040.000000,699.000000,276.000000


In [489]:
import pandas as pd
import plotly.express as px

# Carregar os dados
df = pd.read_csv('df_grouped.csv')

# Verificar se o nome da coluna está correto (se necessário, ajuste o nome da coluna)
df.columns  # Verifique os nomes das colunas para garantir que 'DNA fragmentation index, DFI (%)' está correto

# Definir as faixas de FSH
bins = [0, 18.5, 25, 30, 35, 40, float('inf')]
labels = ['0-18,5', '18,5-24,9', '25-29,9', '30-34,9', '35-39,9', '40+']
df['FSH_Group'] = pd.cut(df['Body mass index (kg/m²)'], bins=bins, labels=labels, right=False)

# Somar os tipos de espermatozoides por faixa de FSH
df_fsh_group = df.groupby('FSH_Group')[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum().reset_index()

# Calcular o total de espermatozoides por faixa de FSH
df_fsh_group['Total'] = df_fsh_group[['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']].sum(axis=1)

# Calcular a proporção de cada tipo
for col in ['Tipo_A', 'Tipo_B', 'Tipo_C', 'Tipo_D', 'Hiperativo']:
    df_fsh_group[f'{col}_Proporcao'] = df_fsh_group[col] / df_fsh_group['Total']

# Selecionar apenas as colunas de proporção para o gráfico
df_proporcao = df_fsh_group[['FSH_Group', 'Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao']]

# Criar o gráfico de barras empilhadas com proporções
fig = px.bar(df_proporcao, 
             x='FSH_Group', 
             y=['Tipo_A_Proporcao', 'Tipo_B_Proporcao', 'Tipo_C_Proporcao', 'Tipo_D_Proporcao', 'Hiperativo_Proporcao'], 
             title="Proporção de tipos de espermatozoides por Body mass index (kg/m²)",
             labels={"FSH_Group": "Body mass index (kg/m²)", "value": "Proporção", "variable": "Tipo"},
             barmode='stack')

# Mostrar o gráfico
fig.show()


C:\Users\ccana\AppData\Local\Temp\ipykernel_23064\259361618.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [375]:
# Análise de variabilidade dos tipos de espermatozoides
type_variability = df.groupby('tipo_espermatozoide')[['VCL', 'VAP', 'VSL']].agg(['mean', 'median'])
print(df.groupby(['tipo_espermatozoide']).size())
print(type_variability)


tipo_espermatozoide
Tipo A               152
Tipo B             15292
Tipo C             14661
Tipo D             11193
Tipo Hiperativo     6706
dtype: int64
                            VCL                     VAP              \
                           mean      median        mean      median   
tipo_espermatozoide                                                   
Tipo A               197.337230  150.008160  185.343030  143.399960   
Tipo B               144.567597  132.307500  139.878541  128.568386   
Tipo C               189.936541  155.713220  153.737375  124.779679   
Tipo D                 0.000000    0.000000    0.000000    0.000000   
Tipo Hiperativo       61.873672   51.901003   57.578387   49.923700   

                            VSL              
                           mean      median  
tipo_espermatozoide                          
Tipo A               172.299282  131.739320  
Tipo B                86.690348   75.475063  
Tipo C               124.047452   99.937514

In [376]:
df_encoded = pd.get_dummies(df, columns=['tipo_espermatozoide'], drop_first=False)
df_encoded.columns

Index(['ID', 'tracker_id', 'x', 'y', 'VCL', 'VSL', 'VAP', 'ALH', 'MAD',
       'Unnamed: 0', 'Serum C14:0 (myristic acid)',
       'Serum C16:0 (palmitic acid)', 'Serum C16:1 (palmitoleic acid)',
       'Serum C18:0 (stearic acid)', 'Serum C18:1 n-9 (oleic acid)',
       'Serum total C18:1', 'Serum C18:2 n-6 (linoleic acid, LA)',
       'Serum C18:3 n-6 (gamma-linoleic acid, GLA)', 'Serum C20:1 n-9',
       'Serum C20:2 n-6', 'Serum C20:3 n-6', 'Serum C20:4 n-6',
       'Serum C20:5 n-3  (eicosapentaenoic acid, EPA)',
       'Serum C22:5 n-3 (docosapentaenoic acid, DPA)',
       'Serum C22:6 n-3 (docosahexaenoic acid, DHA)',
       'Sperm C14:0 (myristic acid)', 'Sperm C15:0 (pentadecanoic acid)',
       'Sperm C16:0 (palmitic acid)', 'Sperm C16:1 n-7 (palmitoleic acid)',
       'Sperm C17:0', 'Sperm C18:0 (stearic acid)',
       'Sperm C18:1 trans n-6 to n-11', 'Sperm C18:1 n-9 (oleic acid)',
       'Sperm C18:1 n-7 to n-11', 'Sperm C18:2 n-6 (Linoleic acid, LA)',
       'Sperm C20:0'

In [377]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import pandas as pd
import numpy as np


In [404]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Codificação da variável-alvo
label_encoder = LabelEncoder()
df['tipo_espermatozoide'] = label_encoder.fit_transform(df['tipo_espermatozoide'])

# Separando atributos e alvo
X = df.drop(['tipo_espermatozoide', 'VCL', 'VAP', 'VSL', 'MAD', 'ALH', 'ID', 'tracker_id', 'x', 'y', 'Unnamed: 0'], axis=1)
y = df['tipo_espermatozoide']

# Dividindo em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo de regressão
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

# Predição
y_pred = model.predict(X_test)

# Cálculo das métricas de avaliação
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE (Erro Médio Quadrático): {mse:.2f}")
print(f"RMSE (Raiz do Erro Médio Quadrático): {rmse:.2f}")
print(f"MAE (Erro Médio Absoluto): {mae:.2f}")
print(f"R² (Coeficiente de Determinação): {r2:.2f}")

# Importância das variáveis
importances = model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances * 100
}).sort_values(by='Importance', ascending=False)

# Exibir as 10 variáveis mais importantes
print(importance_df.head(10))


MSE (Erro Médio Quadrático): 3.93
RMSE (Raiz do Erro Médio Quadrático): 1.98
MAE (Erro Médio Absoluto): 1.80
R² (Coeficiente de Determinação): 0.01
                                       Feature  Importance
37  Sperm C22:6,n3 (docosahexaenoic acid, DHA)   32.292562
53                          Immotile sperm (%)   27.058415
51                    Progressive motility (%)    8.136468
7   Serum C18:3 n-6 (gamma-linoleic acid, GLA)    7.213422
47               Midpiece and neck defects (%)    3.271816
2               Serum C16:1 (palmitoleic acid)    2.937124
9                              Serum C20:2 n-6    2.720063
41               Sperm concentration (x10⁶/mL)    2.246908
45                      Normal spermatozoa (%)    2.024975
23                     Sperm C18:1 n-7 to n-11    1.841629


In [424]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# Separando os atributos (X) e a variável-alvo (y)
def importance_feature_type(f):
    features = ['tipo_espermatozoide_Tipo A', 'tipo_espermatozoide_Tipo B',
       'tipo_espermatozoide_Tipo C', 'tipo_espermatozoide_Tipo D',
       'tipo_espermatozoide_Tipo Hiperativo']
    X = df_encoded.drop(['VCL', 'VAP', 'VSL', 'MAD', 'ALH', 'ID', 'tracker_id', 'x', 'y', 'Unnamed: 0'] + features, axis=1)
    y = df_encoded[f]

    # Dividindo os dados em treino e teste
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)

    # Predição
    y_pred = model.predict(X_test)

    # Cálculo das métricas de classificação
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    print(f"Classificação para {f}")
    print(f"Acurácia: {acc:.2f}")
    print("Matriz de Confusão:")
    print(cm)
    print("Relatório de Classificação:")
    print(report)

    # Importância das variáveis
    importances = model.feature_importances_
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importances * 100
    }).sort_values(by='Importance', ascending=False)

    # Exibir as 10 variáveis mais importantes
    print("Variáveis mais importantes:")
    print(importance_df.head(10))

importance_feature_type('tipo_espermatozoide_Tipo A')
importance_feature_type('tipo_espermatozoide_Tipo B')
importance_feature_type('tipo_espermatozoide_Tipo C')
importance_feature_type('tipo_espermatozoide_Tipo D')
importance_feature_type('tipo_espermatozoide_Tipo Hiperativo')

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Classificação para tipo_espermatozoide_Tipo A
Acurácia: 1.00
Matriz de Confusão:
[[9578    0]
 [  23    0]]
Relatório de Classificação:
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      9578
        True       0.00      0.00      0.00        23

    accuracy                           1.00      9601
   macro avg       0.50      0.50      0.50      9601
weighted avg       1.00      1.00      1.00      9601

Variáveis mais importantes:
                                   Feature  Importance
32  Sperm C20:4 n-6 and C22:1 n-9 combined   14.788104
54          High DNA stainability, HDS (%)   10.440000
23                 Sperm C18:1 n-7 to n-11    9.880224
1              Serum C16:0 (palmitic acid)    9.860830
10                         Serum C20:3 n-6    9.275782
8                          Serum C20:1 n-9    8.603718
48                        Tail defects (%)    6.354290
17             Sperm C16:0 (palmitic acid)    4.334304
30            

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no pred

Classificação para tipo_espermatozoide_Tipo B
Acurácia: 0.67
Matriz de Confusão:
[[6471    0]
 [3130    0]]
Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.67      1.00      0.81      6471
        True       0.00      0.00      0.00      3130

    accuracy                           0.67      9601
   macro avg       0.34      0.50      0.40      9601
weighted avg       0.45      0.67      0.54      9601

Variáveis mais importantes:
                                              Feature  Importance
53                                 Immotile sperm (%)   23.332209
27             Sperm C18:3 n-3 (a-linoleic acid, ALA)   12.692718
4                        Serum C18:1 n-9 (oleic acid)   12.514646
33       Sperm C20:5 n-3 (eicosapentaenoic acid, EPA)    9.461571
12      Serum C20:5 n-3  (eicosapentaenoic acid, EPA)    8.521302
10                                    Serum C20:3 n-6    6.151800
55                   DNA fragmentation index, D

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Classificação para tipo_espermatozoide_Tipo C
Acurácia: 0.69
Matriz de Confusão:
[[6646    0]
 [2955    0]]
Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.69      1.00      0.82      6646
        True       0.00      0.00      0.00      2955

    accuracy                           0.69      9601
   macro avg       0.35      0.50      0.41      9601
weighted avg       0.48      0.69      0.57      9601

Variáveis mais importantes:
                                       Feature  Importance
48                            Tail defects (%)   30.562539
28                             Sperm C20:1 n-9   13.323408
38                       Abstinence time(days)   12.313293
52          Non progressive sperm motility (%)    8.242513
37  Sperm C22:6,n3 (docosahexaenoic acid, DHA)    7.182206
53                          Immotile sperm (%)    4.432357
8                              Serum C20:1 n-9    4.194251
18          Sperm C16:1 n-7 (palmitol

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

C:\Users\ccana\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Classificação para tipo_espermatozoide_Tipo D
Acurácia: 0.77
Matriz de Confusão:
[[7401    0]
 [2200    0]]
Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.77      1.00      0.87      7401
        True       0.00      0.00      0.00      2200

    accuracy                           0.77      9601
   macro avg       0.39      0.50      0.44      9601
weighted avg       0.59      0.77      0.67      9601

Variáveis mais importantes:
                                       Feature  Importance
40                                 Age (years)   18.267038
49                     Cytoplasmic droplet (%)   13.512394
51                    Progressive motility (%)   12.428707
3                   Serum C18:0 (stearic acid)   11.377347
31                             Sperm C20:3 n-6    7.969430
32      Sperm C20:4 n-6 and C22:1 n-9 combined    7.724033
4                 Serum C18:1 n-9 (oleic acid)    4.613854
42                    Total sperm cou

In [ ]:
df_encoded[df_encoded['tipo_espermatozoide_Tipo Hiperativo'] == 1].mean(axis=0).to_frame().T[['Abstinence time(days)',
       'Body mass index (kg/m²)', 'Age (years)',
       'Sperm concentration (x10⁶/mL)', 'Total sperm count (x10⁶)',
       'Ejaculate volume (mL)', 'Sperm vitality (%)', 'Normal spermatozoa (%)',
       'Head defects (%)', 'Midpiece and neck defects (%)', 'Tail defects (%)',
       'Cytoplasmic droplet (%)', 'Teratozoospermia index',
       'Progressive motility (%)', 'Non progressive sperm motility (%)',
       'Immotile sperm (%)', 'High DNA stainability, HDS (%)',
       'DNA fragmentation index, DFI (%)',
       'Seminal plasma anti-Müllerian hormone (AMH) (pmol/L)',
       'Serum total testosterone (nmol/L)', 'Serum oestradiol (nmol/L)',
       'Serum sex hormone-binding globulin, SHBG (nmol/L)',
       'Serum follicle-stimulating hormone, FSH (IU/L)',
       'Serum Luteinizing hormone, LH (IU/L)', 'Serum inhibin B (ng/L)',
       'Serum anti-Müllerian hormone, AMH (pmol/L)']]

,Abstinence time(days),Body mass index (kg/m²),Age (years),Sperm concentration (x10⁶/mL),Total sperm count (x10⁶),Ejaculate volume (mL),Sperm vitality (%),Normal spermatozoa (%),Head defects (%),Midpiece and neck defects (%),...,"High DNA stainability, HDS (%)","DNA fragmentation index, DFI (%)",Seminal plasma anti-Müllerian hormone (AMH) (pmol/L),Serum total testosterone (nmol/L),Serum oestradiol (nmol/L),"Serum sex hormone-binding globulin, SHBG (nmol/L)","Serum follicle-stimulating hormone, FSH (IU/L)","Serum Luteinizing hormone, LH (IU/L)",Serum inhibin B (ng/L),"Serum anti-Müllerian hormone, AMH (pmol/L)"
0,4.207079,29.307575,36.941992,115.334879,404.209395,3.86779,85.805547,3.631211,95.460394,23.282225,...,8.641366,19.507754,520.206084,17.826051,0.138469,32.819415,3.868059,3.778823,206.200567,54.97122


In [440]:
df_encoded[df_encoded['tipo_espermatozoide_Tipo B'] == 1].mean(axis=0).to_frame().T[['Abstinence time(days)',
       'Body mass index (kg/m²)', 'Age (years)',
       'Sperm concentration (x10⁶/mL)', 'Total sperm count (x10⁶)',
       'Ejaculate volume (mL)', 'Sperm vitality (%)', 'Normal spermatozoa (%)',
       'Head defects (%)', 'Midpiece and neck defects (%)', 'Tail defects (%)',
       'Cytoplasmic droplet (%)', 'Teratozoospermia index',
       'Progressive motility (%)', 'Non progressive sperm motility (%)',
       'Immotile sperm (%)', 'High DNA stainability, HDS (%)',
       'DNA fragmentation index, DFI (%)',
       'Seminal plasma anti-Müllerian hormone (AMH) (pmol/L)',
       'Serum total testosterone (nmol/L)', 'Serum oestradiol (nmol/L)',
       'Serum sex hormone-binding globulin, SHBG (nmol/L)',
       'Serum follicle-stimulating hormone, FSH (IU/L)',
       'Serum Luteinizing hormone, LH (IU/L)', 'Serum inhibin B (ng/L)',
       'Serum anti-Müllerian hormone, AMH (pmol/L)']]

,Abstinence time(days),Body mass index (kg/m²),Age (years),Sperm concentration (x10⁶/mL),Total sperm count (x10⁶),Ejaculate volume (mL),Sperm vitality (%),Normal spermatozoa (%),Head defects (%),Midpiece and neck defects (%),...,"High DNA stainability, HDS (%)","DNA fragmentation index, DFI (%)",Seminal plasma anti-Müllerian hormone (AMH) (pmol/L),Serum total testosterone (nmol/L),Serum oestradiol (nmol/L),"Serum sex hormone-binding globulin, SHBG (nmol/L)","Serum follicle-stimulating hormone, FSH (IU/L)","Serum Luteinizing hormone, LH (IU/L)",Serum inhibin B (ng/L),"Serum anti-Müllerian hormone, AMH (pmol/L)"
0,3.905284,28.065609,35.33063,132.975484,485.690708,4.001824,90.137065,4.097973,94.994121,22.308357,...,8.531847,13.88203,441.866793,17.794755,0.135566,31.962726,3.432359,3.517198,217.577622,57.620913
